# Hallmark Command Line Demo: add, commit, checkout, and status


## Setup


In [ ]:
rm -rf repo1
rm -rf repo2.hm
mkdir repo1


## 1. Initialize the hallmark repo


In [ ]:
hallmark init repo1


In [ ]:
echo "=== .hm directory ==="
ls repo1/.hm/
echo ""
echo "=== initial config.yml ==="
cat repo1/.hm/config.yml
echo ""
echo "=== objects stored so far ==="
find repo1/.hm/objects -type f 2>/dev/null | wc -l


## 2. Create data files on `main`


In [ ]:
pushd repo1
for f in a{0,0.75}_i{0,30,60,90,120}.h5; do echo "$f" > "$f"; done
echo "Files in repo1/:"
ls *.h5
popd


## 3. Add files and commit on `main`


In [ ]:
pushd repo1 && hallmark add "a{a}_i{i}.h5" && popd


In [ ]:
echo "=== config.yml after add ==="
cat repo1/.hm/config.yml
echo ""
echo "=== data.tsv after add ==="
cat repo1/.hm/data.tsv


In [ ]:
pushd repo1 && hallmark commit -m "add main dataset" && popd


In [ ]:
echo "Objects stored after commit:"
find repo1/.hm/objects -type f | wc -l


### Clean status


In [ ]:
pushd repo1 && hallmark status && popd


### Current add modes

- `hallmark add "a{a}_i{i}.h5"` parses files using the branch fmt and updates `config.yml`.
- `hallmark add .` rebuilds `data.tsv` from files that currently exist using the fmt already set in `config.yml`.
- `hallmark set-config --fmt ...` is the explicit way to change branch fmt before `hallmark add .`.


## 4. Demo: `hallmark add .` removes deleted files from the manifest

We use a temporary branch so the main workflow stays intact.
On this branch we switch to `b*` files first so it is easy to distinguish from `main`.
Because the filename family changes, we update the branch fmt explicitly with `hallmark set-config` before using `hallmark add .`.


In [ ]:
pushd repo1 && hallmark checkout sync-demo && popd


In [ ]:
pushd repo1
rm a*.h5
for f in b{0,0.75}_i{0,30,60,90,120}.h5; do echo "$f" > "$f"; done
echo "Files after switching sync-demo to b files:"
ls *.h5
popd


In [ ]:
pushd repo1
hallmark set-config --fmt "b{a}_i{i}.h5"
echo "config.yml after hallmark set-config:"
cat .hm/config.yml
popd


In [ ]:
pushd repo1
rm b0.75_i{0,30,60,90,120}.h5
echo "Files after deleting some tracked b files:"
ls *.h5
popd


In [ ]:
pushd repo1 && hallmark add . && popd


In [ ]:
echo "Manifest after hallmark add dot:"
cat repo1/.hm/data.tsv


In [ ]:
pushd repo1 && hallmark status && popd


In [ ]:
pushd repo1 && hallmark commit -m "sync manifest with hallmark add dot" && popd


In [ ]:
pushd repo1 && hallmark checkout main && popd


In [ ]:
echo "Files after checkout back to main:"
ls repo1/*.h5


## 5. Checkout a new branch

The new branch inherits the same fmt from `main`.


In [ ]:
pushd repo1 && hallmark checkout experiment && popd


In [ ]:
echo "Current .hm branch:"
git -C repo1/.hm branch --show-current
echo ""
echo "config.yml on experiment:"
cat repo1/.hm/config.yml
echo ""
echo "Files after checkout:"
ls repo1/*.h5


## 6. Replace the experiment branch dataset with new files that still match the same fmt


In [ ]:
pushd repo1
rm a*.h5
for f in a{1,1.5}_i{15,45,75,105,110}.h5; do echo "$f" > "$f"; done
echo "Files after replacing the dataset on experiment:"
ls *.h5
popd


In [ ]:
pushd repo1 && hallmark add . && popd


In [ ]:
pushd repo1 && hallmark status && popd


In [ ]:
echo "experiment data.tsv before commit:"
cat repo1/.hm/data.tsv


In [ ]:
pushd repo1 && hallmark commit -m "replace experiment dataset" && popd


In [ ]:
echo "Objects stored after experiment commit:"
find repo1/.hm/objects -type f | wc -l


## 7. Checkout back to `main`


In [ ]:
pushd repo1 && hallmark checkout main && popd


In [ ]:
echo "Files after checkout main:"
ls repo1/*.h5
echo ""
echo "main data.tsv:"
cat repo1/.hm/data.tsv


## 8. Checkout back to `experiment` to verify


In [ ]:
pushd repo1 && hallmark checkout experiment && popd


In [ ]:
echo "Files after checkout experiment:"
ls repo1/*.h5
echo ""
echo "experiment data.tsv:"
cat repo1/.hm/data.tsv


## 9. Uncommitted changes block checkout


In [ ]:
pushd repo1
echo "a2_i15.h5" > a2_i15.h5
echo "Files after creating a2_i15.h5:"
ls *.h5
popd


In [ ]:
pushd repo1 && hallmark add . && popd


### Dirty status


In [ ]:
pushd repo1 && hallmark status && popd


In [ ]:
pushd repo1
if hallmark checkout main; then
  echo "checkout unexpectedly succeeded"
else
  echo "checkout failed as expected"
fi
popd


To fix this, commit the staged changes first or discard them.
